# AutoEncoder on Correlation Matrices
In this notebook we train an AutoEncoder to reconstruct correlation matrices using non-linear activation functions.

In [ ]:
from pathlib import Path
import copy
import sys
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Step 1: Load Train/Val/Test Splits
Load correlation matrices from an existing dataset folder containing `train.pt`, `val.pt`, and `test.pt`.
- Local PC: `data/processed/dataset/<DATASET_NAME>`
- Google Colab: `dataset_tesi/<DATASET_NAME>`

In [ ]:
DATASET_NAME = 'data_00_20_w724_s10'

if 'google.colab' in sys.modules:
    print('Ambiente rilevato: Google Colab')
    IS_COLAB = True
else:
    print('Ambiente rilevato: Locale (PC)')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    base_dir = Path('/content/drive/MyDrive/dataset_tesi')
else:
    project_root = Path.cwd().resolve().parent
    base_dir = project_root / 'data' / 'processed' / 'dataset'

dataset_dir = base_dir / DATASET_NAME

TRAIN_FILE = dataset_dir / 'train.pt'
VAL_FILE = dataset_dir / 'val.pt'
TEST_FILE = dataset_dir / 'test.pt'

for split_path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    if not split_path.exists():
        raise FileNotFoundError(
            f"Errore: Il file '{split_path.name}' non e stato trovato in: {dataset_dir.absolute()}"
        )

print(f'Dataset selezionato: {DATASET_NAME}')
print(f'Train: {TRAIN_FILE.name} | Val: {VAL_FILE.name} | Test: {TEST_FILE.name}')

In [ ]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')

    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')

    return corr_tensor.float(), meta


train_corr, train_meta = load_corr_payload(TRAIN_FILE)
val_corr, val_meta = load_corr_payload(VAL_FILE)
test_corr, test_meta = load_corr_payload(TEST_FILE)

if train_corr.shape[1:] != test_corr.shape[1:]:
    raise ValueError(f'Train/test asset dims mismatch: {train_corr.shape} vs {test_corr.shape}')
if val_corr.shape[1:] != train_corr.shape[1:]:
    raise ValueError(f'Val/train asset dims mismatch: {val_corr.shape} vs {train_corr.shape}')

print(f'train_corr shape: {tuple(train_corr.shape)}')
print(f'val_corr shape:   {tuple(val_corr.shape)}')
print(f'test_corr shape:  {tuple(test_corr.shape)}')

## Step 2: Vectorize Full Matrices
Flatten each correlation matrix to a full $N \times N$ vector for train/val/test splits.

In [ ]:
train_np = train_corr.numpy().astype(np.float32)
val_np = val_corr.numpy().astype(np.float32)
test_np = test_corr.numpy().astype(np.float32)

n_train, n_assets, _ = train_np.shape
n_val = val_np.shape[0]
n_test = test_np.shape[0]

n_features = n_assets * n_assets

X_train = train_np.reshape(n_train, n_features)
X_val = val_np.reshape(n_val, n_features)
X_test = test_np.reshape(n_test, n_features)

X_train_tensor = torch.from_numpy(X_train)
X_val_tensor = torch.from_numpy(X_val)
X_test_tensor = torch.from_numpy(X_test)

print(f'Train matrices: {n_train}')
print(f'Val matrices: {n_val}')
print(f'Test matrices: {n_test}')
print(f'Assets per matrix: {n_assets}')
print(f'Full-matrix features: {n_features}')

## Step 3: Define and Train AutoEncoder
Train a compact fully-connected AutoEncoder on full-matrix vectors using MSE loss.
A validation split is used to keep the best model checkpoint.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 3, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [4096, 2048, 1024, 512, 256, 128, 64, 32, 16, 8]

        dimensions = [input_dim, *hidden_dims, latent_dim]

        encoder_layers = []
        for i in range(len(dimensions) - 1):
            encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
            if i < len(dimensions) - 2:
                encoder_layers.append(nn.ReLU())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_dims = dimensions[::-1]
        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(nn.ReLU())
            else:
                decoder_layers.append(nn.Tanh())
        self.decoder = nn.Sequential(*decoder_layers)

    def architecture_signature(self):
        return [
            [int(layer.in_features), int(layer.out_features)]
            for layer in self.encoder
            if isinstance(layer, nn.Linear)
        ]

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [ ]:
# Resume training toggle
RUN = 'AE_01'
RESUME_TRAINING = False
RESUME_MODEL_PATH = "best_models_tesi/AE/AE_01/AE_best_AE_01.pt"  # Example: 'best_models_tesi/AE/AE_01/AE_best_AE_01.pt'


LATENT_DIM = 3
EPOCHS = 50
LR = 1e-5
WEIGHT_DECAY = 0
BATCH_SIZE = 8

# Hidden layers for the AE encoder (decoder is mirrored automatically).
# If you resume from an old checkpoint, this must match that checkpoint architecture.
HIDDEN_DIMS = [4096, 1024, 256, 64, 32, 16, 8]

# Scheduler selector: 'none', 'cosine', 'step', 'plateau'
SCHEDULER_NAME = 'cosine'

# CosineAnnealingLR params
COSINE_T_MAX = EPOCHS
COSINE_ETA_MIN = 1e-8

# StepLR params
STEP_SIZE = 10
STEP_GAMMA = 0.5

# ReduceLROnPlateau params
PLATEAU_FACTOR = 0.5
PLATEAU_PATIENCE = 5
PLATEAU_THRESHOLD = 1e-3
PLATEAU_MIN_LR = 1e-8

if IS_COLAB:
    run_dir = Path('/content/drive/MyDrive/best_models_tesi/AE')
else:
    run_dir = project_root / 'models' / 'best_models_tesi' / 'AE'

run_output_dir = run_dir / RUN
run_output_dir.mkdir(parents=True, exist_ok=True)
print(f'Results directory: {run_output_dir}')
print(f'Scheduler: {SCHEDULER_NAME}')
print(f'Resume training: {RESUME_TRAINING}')
print(f'Hidden dims: {HIDDEN_DIMS}')

In [ ]:
n_train = X_train_tensor.shape[0]
n_val = X_val_tensor.shape[0]
n_test = X_test_tensor.shape[0]
n_total = n_train + n_val + n_test

VAL_FRACTION = n_val / n_total
TEST_FRACTION = n_test / n_total

train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {n_train}')
print(f'Validation samples: {n_val}')
print(f'Test samples: {n_test}')

In [ ]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_items = 0

    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            x_hat = model(x_batch)
            loss = criterion(x_hat, x_batch)

            batch_size = x_batch.size(0)
            total_loss += loss.item() * batch_size
            total_items += batch_size

    return total_loss / max(total_items, 1)


def train_model(
    model,
    train_loader,
    val_loader,
    epochs,
    lr,
    weight_decay,
    scheduler_name,
    cosine_t_max,
    cosine_eta_min,
    step_size,
    step_gamma,
    plateau_factor,
    plateau_patience,
    plateau_threshold,
    plateau_min_lr,
    initial_best_val=None,
):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler_key = str(scheduler_name).strip().lower()
    valid_schedulers = {'none', 'cosine', 'step', 'plateau'}
    if scheduler_key not in valid_schedulers:
        raise ValueError(f"Invalid scheduler '{scheduler_name}'. Choose one of {sorted(valid_schedulers)}")

    scheduler = None
    if scheduler_key == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cosine_t_max,
            eta_min=cosine_eta_min,
        )
    elif scheduler_key == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=step_size,
            gamma=step_gamma,
        )
    elif scheduler_key == 'plateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=plateau_factor,
            patience=plateau_patience,
            threshold=plateau_threshold,
            min_lr=plateau_min_lr,
        )

    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_state = copy.deepcopy(model.state_dict())
    if initial_best_val is None:
        best_val = float('inf')
    else:
        best_val = float(initial_best_val)

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        seen = 0

        current_lr = float(optimizer.param_groups[0]['lr'])

        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device)
            optimizer.zero_grad()
            x_hat = model(x_batch)
            loss = criterion(x_hat, x_batch)
            loss.backward()
            optimizer.step()

            batch_size = x_batch.size(0)
            running += loss.item() * batch_size
            seen += batch_size

        train_loss = running / max(seen, 1)
        val_loss = validate_epoch(model, val_loader, criterion)

        if scheduler is not None:
            if scheduler_key == 'plateau':
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(
            f'Epoch {epoch:03d}/{epochs} | lr={current_lr:.2e} | '
            f'train_loss={train_loss:.6f} | val_loss={val_loss:.6f}'
        )

    model.load_state_dict(best_state)
    return model, history, best_val

In [ ]:
model = AutoEncoder(
    input_dim=n_features,
    latent_dim=LATENT_DIM,
    hidden_dims=HIDDEN_DIMS,
).to(device)

resume_source_path = None
resume_initial_val = None
if RESUME_TRAINING:
    if RESUME_MODEL_PATH is None:
        raise ValueError('RESUME_TRAINING=True but RESUME_MODEL_PATH is None.')

    resume_source_path = Path(RESUME_MODEL_PATH)
    if not resume_source_path.is_absolute():
        if IS_COLAB:
            resume_source_path = Path('/content/drive/MyDrive') / resume_source_path
        else:
            resume_source_path = project_root / resume_source_path
    resume_source_path = resume_source_path.resolve()

    if not resume_source_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {resume_source_path}')

    checkpoint = torch.load(resume_source_path, map_location=device)
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint

    # Validate architecture compatibility before loading weights to get a clear error message.
    current_shapes = {k: tuple(v.shape) for k, v in model.state_dict().items()}
    loaded_shapes = {k: tuple(v.shape) for k, v in state_dict.items()}

    missing_keys = [k for k in current_shapes.keys() if k not in loaded_shapes]
    shape_mismatches = [
        (k, loaded_shapes[k], current_shapes[k])
        for k in current_shapes.keys()
        if k in loaded_shapes and loaded_shapes[k] != current_shapes[k]
    ]

    if missing_keys or shape_mismatches:
        suggested_hidden = []
        idx = 0
        while True:
            key = f'encoder.{idx}.weight'
            if key not in state_dict:
                break
            w = state_dict[key]
            if hasattr(w, 'shape') and len(w.shape) == 2:
                suggested_hidden.append(int(w.shape[0]))
            idx += 2  # encoder is Linear, ReLU, Linear, ReLU, ...

        if suggested_hidden:
            # Last encoder layer is latent_dim; hidden dims are all previous ones.
            ckpt_latent_dim = suggested_hidden[-1]
            ckpt_hidden_dims = suggested_hidden[:-1]
            hint = (
                f'Checkpoint expects latent_dim={ckpt_latent_dim}, '
                f'hidden_dims={ckpt_hidden_dims}. '
                'Set these values to match before resuming, or start a new run without resume.'
            )
        else:
            hint = 'Checkpoint architecture could not be inferred. Use a checkpoint saved with the same model architecture.'

        details = []
        if missing_keys:
            details.append(f'Missing keys (first 5): {missing_keys[:5]}')
        if shape_mismatches:
            k, loaded_shape, current_shape = shape_mismatches[0]
            details.append(f'Shape mismatch example: {k} loaded={loaded_shape}, current={current_shape}')

        raise RuntimeError(
            'Incompatible checkpoint for current AutoEncoder architecture. ' + hint + ' | ' + ' ; '.join(details)
        )

    model.load_state_dict(state_dict)
    resume_initial_val = validate_epoch(model, val_loader, nn.MSELoss())
    print(f'Loaded checkpoint: {resume_source_path}')
    print(f'Checkpoint validation MSE (baseline for resume): {resume_initial_val:.6f}')

model, history, best_val = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    scheduler_name=SCHEDULER_NAME,
    cosine_t_max=COSINE_T_MAX,
    cosine_eta_min=COSINE_ETA_MIN,
    step_size=STEP_SIZE,
    step_gamma=STEP_GAMMA,
    plateau_factor=PLATEAU_FACTOR,
    plateau_patience=PLATEAU_PATIENCE,
    plateau_threshold=PLATEAU_THRESHOLD,
    plateau_min_lr=PLATEAU_MIN_LR,
    initial_best_val=resume_initial_val,
)

best_model_path = run_output_dir / f'AE_best_{RUN}.pt'
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_config': {
            'input_dim': int(n_features),
            'latent_dim': int(LATENT_DIM),
            'hidden_dims': [int(x) for x in HIDDEN_DIMS],
            'architecture_signature': model.architecture_signature(),
        },
    },
    best_model_path,
)
print(f'Saved best model: {best_model_path}')

test_loss = validate_epoch(model, test_loader, nn.MSELoss())

print(f'\nBest validation MSE: {best_val:.6f}')
print(f'Test MSE: {test_loss:.6f}')

## Step 4: Reconstruct Correlation Matrices and Compute Errors
Decode full-matrix vectors, reshape them back to $N \times N$, and evaluate reconstruction quality.

In [ ]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats


# Reconstruction on test set
model.eval()
with torch.no_grad():
    X_recon = model(X_test_tensor.to(device)).cpu().numpy()

orig_corr = test_np
recon_corr = X_recon.reshape(-1, n_assets, n_assets).astype(np.float32)

errors_df, errors_stats = reconstruction_errors(orig_corr, recon_corr)

print('Test-set reconstruction error statistics:')
display(errors_stats)

In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

sample_idx = 0

if not (0 <= sample_idx < orig_corr.shape[0]):
    raise ValueError(f'sample_idx must be in [0, {orig_corr.shape[0] - 1}]')

orig_mat = orig_corr[sample_idx]
recon_mat = recon_corr[sample_idx]
err_mat = recon_mat - orig_mat

mse_sample = float(np.mean((recon_mat - orig_mat) ** 2))
mae_sample = float(np.mean(np.abs(recon_mat - orig_mat)))
fro_sample = float(np.linalg.norm(recon_mat - orig_mat, ord='fro'))

print(f'Sample index (test set): {sample_idx}')
print(f'MSE: {mse_sample:.6f} | MAE: {mae_sample:.6f} | Frobenius: {fro_sample:.6f}')

heatmap_base_dir = run_output_dir / 'heatmaps'
heatmap_base_dir.mkdir(parents=True, exist_ok=True)

# Create subdirectory for this sample index
heatmap_dir = heatmap_base_dir / f'idx{sample_idx}'
heatmap_dir.mkdir(parents=True, exist_ok=True)

# Fixed color scale shared by original, reconstructed, and error heatmaps.
scale_vmin, scale_vmax = -1, 1

# Hierarchical clustering heatmaps
cluster_true = sns.clustermap(
    orig_mat,
    vmin=scale_vmin,
    vmax=scale_vmax,
    cmap='coolwarm',
    linewidths=0,
    figsize=(8, 8),
    cbar_kws={'label': 'Correlation'},
)
cluster_true.figure.suptitle('Original Matrix (Hierarchical Clustering)', y=1.02)
path_cluster_orig = heatmap_dir / f'ae_cluster_original.png'
cluster_true.figure.savefig(path_cluster_orig, dpi=150, bbox_inches='tight')
plt.show()

row_linkage = cluster_true.dendrogram_row.linkage
col_linkage = cluster_true.dendrogram_col.linkage

cluster_recon = sns.clustermap(
    recon_mat,
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    vmin=scale_vmin,
    vmax=scale_vmax,
    cmap='coolwarm',
    linewidths=0,
    figsize=(8, 8),
    cbar_kws={'label': 'Correlation'},
)
cluster_recon.figure.suptitle('Reconstructed Matrix (Same Cluster Order)', y=1.02)
path_cluster_recon = heatmap_dir / f'ae_cluster_reconstructed.png'
cluster_recon.figure.savefig(path_cluster_recon, dpi=150, bbox_inches='tight')
plt.show()

cluster_err = sns.clustermap(
    err_mat,
    row_linkage=row_linkage,
    col_linkage=col_linkage,
    center=0,
    vmin=scale_vmin,
    vmax=scale_vmax,
    cmap='coolwarm',
    linewidths=0,
    figsize=(8, 8),
    cbar_kws={'label': 'Error (recon - original)'},
)
cluster_err.figure.suptitle('Error Matrix (Same Cluster Order)', y=1.02)
path_cluster_err = heatmap_dir / f'ae_cluster_error.png'
cluster_err.figure.savefig(path_cluster_err, dpi=150, bbox_inches='tight')
plt.show()

# Standard heatmaps (no clustering, original matrix order)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(orig_mat, ax=axes[0], vmin=scale_vmin, vmax=scale_vmax, cmap='coolwarm', square=True)
axes[0].set_title('Original Matrix (No Clustering)')

sns.heatmap(recon_mat, ax=axes[1], vmin=scale_vmin, vmax=scale_vmax, cmap='coolwarm', square=True)
axes[1].set_title('Reconstructed Matrix (No Clustering)')

sns.heatmap(err_mat, ax=axes[2], vmin=scale_vmin, vmax=scale_vmax, cmap='coolwarm', center=0, square=True)
axes[2].set_title('Error Matrix (No Clustering)')

for ax in axes:
    ax.set_xlabel('Assets')
    ax.set_ylabel('Assets')

plt.tight_layout()
path_standard = heatmap_dir / f'ae_heatmaps_standard.png'
fig.savefig(path_standard, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved heatmaps for matrix index {sample_idx}:')
print(f' - {path_cluster_orig}')
print(f' - {path_cluster_recon}')
print(f' - {path_cluster_err}')
print(f' - {path_standard}')

## Step 5: Visual Diagnostics
Show training curves and the distribution of per-matrix MSE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_idx = np.arange(1, len(history['train_loss']) + 1)
start_plot = 5  # Skip first 5 epochs

train_loss_plot = np.asarray(history['train_loss'])[start_plot:]
val_loss_plot = np.asarray(history['val_loss'])[start_plot:]
epochs_plot = epochs_idx[start_plot:]

axes[0].plot(epochs_plot, train_loss_plot, label='Train Loss', linewidth=2)
axes[0].plot(epochs_plot, val_loss_plot, label='Validation Loss', linewidth=2)
axes[0].set_title('Training Curves', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].hist(errors_df['MSE'], bins=30, alpha=0.7, color='#1f77b4', edgecolor='black')
axes[1].set_title('Distribution of Reconstruction MSE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('MSE per matrix')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

plt.tight_layout()
diag_path = run_output_dir / f'ae_diagnostics_{RUN}.png'
fig.savefig(diag_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved: {diag_path}')

In [ ]:
results_path = run_output_dir / f'ae_results_{RUN}.json'

scheduler_key = str(SCHEDULER_NAME).strip().lower()
scheduler_label_map = {
    'none': None,
    'cosine': 'CosineAnnealingLR',
    'step': 'StepLR',
    'plateau': 'ReduceLROnPlateau',
}

scheduler_payload = {
    'key': scheduler_key,
    'type': scheduler_label_map.get(scheduler_key, 'Unknown'),
    'enabled': scheduler_key != 'none',
}

if scheduler_key == 'cosine':
    scheduler_payload.update({
        'T_max': int(COSINE_T_MAX),
        'eta_min': float(COSINE_ETA_MIN),
    })
elif scheduler_key == 'step':
    scheduler_payload.update({
        'step_size': int(STEP_SIZE),
        'gamma': float(STEP_GAMMA),
    })
elif scheduler_key == 'plateau':
    scheduler_payload.update({
        'factor': float(PLATEAU_FACTOR),
        'patience': int(PLATEAU_PATIENCE),
        'threshold': float(PLATEAU_THRESHOLD),
        'min_lr': float(PLATEAU_MIN_LR),
    })

results_payload = {
    'run': RUN,
    'dataset_name': DATASET_NAME,
    'dataset_dir': str(dataset_dir),
    'data': {
        'n_matrices': int(n_total),
        'n_assets': int(n_assets),
        'n_features': int(n_features),
        'splits': {
            'train_size': int(n_train),
            'val_size': int(n_val),
            'test_size': int(n_test),
            'val_fraction': float(VAL_FRACTION),
            'test_fraction': float(TEST_FRACTION),
        },
    },
    'model': {
        'latent_dim': int(LATENT_DIM),
        'hidden_dims': [int(x) for x in HIDDEN_DIMS],
        'epochs': int(EPOCHS),
        'lr': float(LR),
        'weight_decay': float(WEIGHT_DECAY),
        'batch_size': int(BATCH_SIZE),
        'resume': {
            'enabled': bool(RESUME_TRAINING),
            'source_model_path': str(resume_source_path) if resume_source_path is not None else None,
        },
        'scheduler': scheduler_payload,
    },
    'artifacts': {
        'best_model_path': str(best_model_path),
    },
    'metrics': {
        'best_validation_mse': float(best_val),
        'test_mse': float(test_loss),
        'reconstruction_stats': errors_stats.to_dict(),
    },
    'loss_history': {
        'train_loss': [float(x) for x in history['train_loss']],
        'val_loss': [float(x) for x in history['val_loss']],
        'lr': [float(x) for x in history['lr']],
    },
    'reconstruction_errors_per_matrix': {
        'MSE': [float(x) for x in errors_df['MSE'].tolist()],
        'MAE': [float(x) for x in errors_df['MAE'].tolist()],
        'Frobenius': [float(x) for x in errors_df['Frobenius'].tolist()],
    },
}

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results_payload, f, indent=4)

print(f'Saved results: {results_path}')